In [ ]:
import os
import tarfile
import urllib.request
import pandas as pd

DATASET_URL  = "https://github.com/uvicrepo/dataset-deadrepo/releases/download/v1.0/datasets.tar.gz"
DATASET_PATH = os.path.join("datasets")
FLOW_PATH    = os.path.join("datasets", "flowFeatures_datasets")
PACKET_PATH  = os.path.join("datasets", "packetFeatures_datasets")

FLOW_FILES = {
    "benign":       ["BenignTraffic.pcap_Flow.csv",
                    "BenignTraffic1.pcap_Flow.csv",
                    "BenignTraffic2.pcap_Flow.csv",
                    "BenignTraffic3.pcap_Flow.csv"],
    "ddos_http":    ["DDoS-HTTP_Flood-.pcap_Flow.csv"],
    "dos_http":     ["DoS-HTTP_Flood.pcap_Flow.csv",
                    "DoS-HTTP_Flood1.pcap_Flow.csv"],
    "dns_spoofing": ["DNS_Spoofing.pcap_Flow.csv"],
    "xss":          ["XSS.pcap_Flow.csv"],
    "brute_force":  ["DictionaryBruteForce.pcap_Flow.csv"],
}

PACKET_FILES = {
    "benign":       ["BenignTraffic.csv",
                    "BenignTraffic1.csv",
                    "BenignTraffic2.csv",
                    "BenignTraffic3.csv"],
    "ddos_http":    ["DDoS-HTTP_Flood-.csv"],
    "dos_http":     ["DoS-HTTP_Flood.csv",
                    "DoS-HTTP_Flood1.csv"],
    "dns_spoofing": ["DNS_Spoofing.csv"],
    "xss":          ["XSS.csv"],
    "brute_force":  ["DictionaryBruteForce.csv"],
}

def _parquet_path(csv_path):
    return csv_path.rsplit(".csv", 1)[0] + ".parquet"

def _convert_to_parquet():
    all_csvs = (
        [os.path.join(FLOW_PATH, f)   for files in FLOW_FILES.values()   for f in files] +
        [os.path.join(PACKET_PATH, f) for files in PACKET_FILES.values() for f in files]
    )
    to_convert = [p for p in all_csvs if not os.path.isfile(_parquet_path(p))]
    if not to_convert:
        return
    print(f"Converting {len(to_convert)} CSV file(s) to Parquet...")
    for csv_path in to_convert:
        print(f"  {os.path.basename(csv_path)}")
        df = pd.read_csv(csv_path, low_memory=False)
        df.to_parquet(_parquet_path(csv_path), index=False)
        del df
    print("Done.")

def fetch_datasets(dataset_url=DATASET_URL, dataset_path=DATASET_PATH):
    if not os.path.isdir(FLOW_PATH):
        tgz_path = os.path.join("datasets", "datasets.tar.gz")
        os.makedirs("datasets", exist_ok=True)
        print("Downloading dataset...")
        urllib.request.urlretrieve(dataset_url, tgz_path)
        print("Extracting...")
        dataset_tgz = tarfile.open(tgz_path)
        dataset_tgz.extractall(path=".")
        dataset_tgz.close()
        os.remove(tgz_path)
    else:
        print("Datasets already downloaded.")
    _convert_to_parquet()

def _load(files_dict, path, key):
    filenames = files_dict[key]
    frames = [pd.read_parquet(_parquet_path(os.path.join(path, f))) for f in filenames]
    return pd.concat(frames, ignore_index=True) if len(frames) > 1 else frames[0]

def load_flow(key):
    print(f"Loading Flow Dataset: {key}...")
    df = _load(FLOW_FILES, FLOW_PATH, key)
    print(f"Shape: {df.shape}")
    return df

def load_packet(key):
    print(f"Loading Packet Dataset: {key}...")
    df = _load(PACKET_FILES, PACKET_PATH, key)
    print(f"Shape: {df.shape}")
    return df

fetch_datasets()

Extracting...
Done.
Converting 20 CSV file(s) to Parquet...
  BenignTraffic.pcap_Flow.csv
  BenignTraffic1.pcap_Flow.csv
  BenignTraffic2.pcap_Flow.csv
  BenignTraffic3.pcap_Flow.csv
  DDoS-HTTP_Flood-.pcap_Flow.csv
  DoS-HTTP_Flood.pcap_Flow.csv
  DoS-HTTP_Flood1.pcap_Flow.csv
  DNS_Spoofing.pcap_Flow.csv
  XSS.pcap_Flow.csv
  DictionaryBruteForce.pcap_Flow.csv
  BenignTraffic.csv
  BenignTraffic1.csv
  BenignTraffic2.csv
  BenignTraffic3.csv
  DDoS-HTTP_Flood-.csv
  DoS-HTTP_Flood.csv
  DoS-HTTP_Flood1.csv
  DNS_Spoofing.csv
  XSS.csv
  DictionaryBruteForce.csv
Conversion done.


### Lazy Load Datasets
Use the cells below to load specific datasets as needed.
Available keys: `benign`, `ddos_http`, `dos_http`, `dns_spoofing`, `xss`, `brute_force`

Call `load_flow(<key>)` for flow-based features or `load_packet(<key>)` for packet-based features.

Example:
```python
df_xss_flow = load_flow("xss")
df_benign_packet = load_packet("benign")
```

In [ ]:
df_xss_flow = load_flow("xss")
df_benign_packet = load_packet("benign")
print("Done.")

Loading flow dataset: xss...
Shape: (3377, 84)
Loading packet dataset: benign...
Shape: (1182798, 135)
